In [46]:
#
# Custom CSS to display the notebook at full browser width
#
# from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))

<br>

# Natural Language Processing (IV)


<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/ai-eng-nbs-4-master/s-403-404/4.%20Natural%20Language%20Processing%20IV.ipynb
</div>



<br>

## What we'll cover in this notebook

In this notebook we apply NLP techniques to real text classification and clustering tasks. We'll go through:

- **Stopwords** — removing common words that add little meaning
- **Bag of Words** — representing text as word frequency dictionaries
- **N-grams** — capturing short word sequences (bigrams, trigrams)
- **Sentiment classification** — training a Naive Bayes classifier using TextBlob on tweet-style data
- **News clustering** — using Bag of Words + KMeans to group news headlines by topic


<br>

## 3.1 Stopwords

These are common words like "the", "a", "an", "in", etc. that don't carry much meaning on their own. While regex can be used to identify these words using patterns, stopwords are usually removed in a separate pre-processing step before applying NLP techniques like bag-of-words and n-grams.

In [47]:
#nltk.download()   # try to use this is you need some package
from nltk.corpus import stopwords
print (stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [48]:
from nltk.corpus import stopwords

def bag_of_words(words):
        return dict([(word, True) for word in words])

def bag_of_words_not_in_set(words, badwords):
    return bag_of_words(set(words) - set(badwords))

def bag_of_non_stopwords(words, stopfile = 'english'):
    badwords = stopwords.words(stopfile)
    return bag_of_words_not_in_set(words, badwords)


bag_of_non_stopwords(['this','is','awesome'])


{'awesome': True}

In [49]:
example = ["my", "tailor", "is", "rich"]

result = bag_of_non_stopwords(example)

print(example)
print(result)

['my', 'tailor', 'is', 'rich']
{'rich': True, 'tailor': True}


<br>

## 3.2 Bag of Words

**Bag of Words (BoW)** is one of the simplest and most widely used ways to represent text as numbers so that machine learning models can work with it.

### The core idea

Each document is represented as a **vector of word counts** (or word presence flags). You build a vocabulary of all unique words seen in your corpus, and then for each document, you count how many times each vocabulary word appears. The resulting vector is the document's BoW representation.

**Example:**

| Document | "cat" | "sat" | "mat" | "the" | "dog" |
|----------|-------|-------|-------|-------|-------|
| "the cat sat on the mat" | 1 | 1 | 1 | 2 | 0 |
| "the dog sat on the mat" | 0 | 1 | 1 | 2 | 1 |

### Key properties

- **Order doesn't matter** — "dog bites man" and "man bites dog" produce identical BoW vectors. This is the main limitation.
- **Sparse vectors** — real corpora have thousands of unique words, so most entries are zero.
- **Interpretable** — each dimension directly corresponds to a word, making it easy to inspect.

### Two common variants

| Variant | What it stores | When to use |
|---------|---------------|-------------|
| **Binary BoW** | `True`/`False` — does the word appear? | Short texts, presence matters more than frequency |
| **Count BoW** | Integer count of each word | Longer documents, frequency carries signal |

### Limitations

- Ignores word order and grammar ("not good" ≠ "good not" in meaning, but identical in BoW).
- Common words ("the", "a") dominate counts → mitigated by **stopword removal** or **TF-IDF** weighting.
- Vocabulary can become very large → controlled with `max_features` in `CountVectorizer`.

<br>

> **Tip:** BoW is a strong baseline. Always try it before reaching for more complex representations like TF-IDF, word embeddings, or transformers.


In [50]:
# First, write a Feature extractor (the following is taken from nltk-trainer package)

# download featx.py (written by Perkins)

from nltk import probability
import math

def bag_of_words(words):
        return dict([(word, True) for word in words])

def bag_of_words_in_set(words, wordset):
        return bag_of_words(set(words) & wordset)
    
def word_counts(words):
        return dict(probability.FreqDist((w for w in words)))

def word_counts_in_set(words, wordset):
        return word_counts((w for w in words if w in wordset))

def train_test_feats(label, instances, featx=bag_of_words, fraction=0.75):
        labeled_instances = [(featx(i), label) for i in instances]
        
        if fraction != 1.0:
                l = len(instances)
                cutoff = int(math.ceil(l * fraction))
                return labeled_instances[:cutoff], labeled_instances[cutoff:]
        else:
                return labeled_instances, labeled_instances

In [51]:
bag_of_words(['this', 'is', 'awesome'])

{'this': True, 'is': True, 'awesome': True}

In [52]:
def bag_of_words_not_in_set(words, badwords):
    return bag_of_words(set(words) - set(badwords))

bag_of_words_not_in_set(['this','is','awesome'],['this'])

{'is': True, 'awesome': True}

In [53]:

# 
# Bag of Words demo
# 
# CountVectorizer from sklearn.feature_extraction.text converts a collection of text documents into a matrix of word counts.
# 

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


# Two example documents (matching the table in the markdown above)
_bow_docs = [
    "the cat sat on the mat", 
    "the dog sat on the mat"
]

# Count BoW with word_counts()
print("Count BoW (word_counts):")
print("Sentence: ", _bow_docs[0])
print(word_counts(_bow_docs[0].split()))

# Count BoW matrix with CountVectorizer
_bow_vect = CountVectorizer()
_bow_matrix = _bow_vect.fit_transform(_bow_docs).toarray()

_bow_df = pd.DataFrame(_bow_matrix,
                        columns=_bow_vect.get_feature_names_out(),
                        index=_bow_docs)
print("\nCount BoW matrix (CountVectorizer):")
_bow_df


Count BoW (word_counts):
Sentence:  the cat sat on the mat
{'the': 2, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1}

Count BoW matrix (CountVectorizer):


,cat,dog,mat,on,sat,the
the cat sat on the mat,1,0,1,1,1,2
the dog sat on the mat,0,1,1,1,1,2


<br>

## 3.3 N-grams

An **N-gram** is a contiguous sequence of *n* items (words, characters, or tokens) from a text. They are one of the simplest ways to capture word order and local context, which plain Bag of Words ignores.

| Name | n | Example (from "the cat sat on the mat") |
|------|---|----------------------------------------|
| Unigram | 1 | `the`; `cat`; `sat`; `on`; `the`; `mat` |
| Bigram | 2 | `the cat`; `cat sat`; `sat on`; `on the`; `the mat`; |
| Trigram | 3 | `the cat sat`; `cat sat on`; `sat on the`; `on the mat` |

### Why N-grams matter

- **BoW loses word order** — "dog bites man" and "man bites dog" produce identical BoW vectors, but different bigram sets.
- **Phrases carry meaning** — bigrams like `"not good"` or `"New York"` mean something that neither word alone conveys.
- **Trade-off** — larger *n* captures more context but creates a much sparser feature space and requires more data to be useful.

### How N-grams are used in NLP pipelines

1. **Text classification** — feed bigram/trigram counts as features to a classifier (e.g., sentiment analysis).
2. **Language modelling** — estimate the probability of the next word given the previous *n-1* words.
3. **Information retrieval** — match multi-word queries against document n-gram indexes.
4. **Spell/grammar checking** — detect unusual n-gram sequences that signal errors.

> **Rule of thumb:** start with unigrams + bigrams. Trigrams and beyond are only worth the added complexity when you have a large, clean corpus.


In [54]:
import nltk
from nltk.util import bigrams

nltk.download("punkt")

# Sample sentence
sentence = "This is a sample sentence for n-gram generation."

# Tokenize the sentence.
tokens = nltk.word_tokenize(sentence.lower())
print(tokens)

# Generate bigrams from the tokens
bigrams = bigrams(tokens)

# Print the bigrams
print(list(bigrams))


['this', 'is', 'a', 'sample', 'sentence', 'for', 'n-gram', 'generation', '.']
[('this', 'is'), ('is', 'a'), ('a', 'sample'), ('sample', 'sentence'), ('sentence', 'for'), ('for', 'n-gram'), ('n-gram', 'generation'), ('generation', '.')]


[nltk_data] Downloading package punkt to /Users/luis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


<br>
<hr>
<hr>
<br>

Discover: Naive Bayes algorithm

1. Do some research: what is the Naive Bayes algorithm + what are common use cases.
2. Watch: Naive Bayes - Explained (5min. video): https://www.youtube.com/watch?v=Kstjz91Ks4U


Time: 10 min

<br>
<hr>
<hr>
<br>

<br>

## 3.4 Training a Naive Bayes Classifier with TextBlob

In this section we build a **tweet sentiment classifier** step by step using [TextBlob](https://textblob.readthedocs.io/en/dev/classifiers.html)'s `NaiveBayesClassifier`. 

<br>


> **Why Naive Bayes?** It assumes each word contributes *independently* to the sentiment probability — a simplification that works surprisingly well on short texts like tweets, and is very fast to train.




In [55]:
# !pip install -U textblob
#!conda install -c https://conda.anaconda.org/sloria textblob -y

In [56]:
from textblob.classifiers import NaiveBayesClassifier

train = [
    ('I love this sandwich.', 'pos'),
    ('This is an amazing place!', 'pos'),
    ('I feel very good about these beers.', 'pos'),
    ('This is my best work.', 'pos'),
    ("What an awesome view", 'pos'),
    ('I do not like this restaurant', 'neg'),
    ('I am tired of this stuff.', 'neg'),
    ("I can't deal with this", 'neg'),
    ('He is my sworn enemy!', 'neg'),
    ('My boss is horrible.', 'neg')
]
test = [
    ('The beer was good.', 'pos'),
    ('I do not enjoy my job', 'neg'),
    ("I ain't feeling dandy today.", 'neg'),
    ("I feel amazing!", 'pos'),
    ('Gary is a friend of mine.', 'pos'),
    ("I can't believe I'm doing this.", 'neg')
]



We create a new classifier by passing training data into the constructor for a NaiveBayesClassifier.



In [57]:
cl = NaiveBayesClassifier(train)
cl.show_informative_features(20)

Most Informative Features
          contains(this) = True              neg : pos    =      2.3 : 1.0
          contains(this) = False             pos : neg    =      1.8 : 1.0
          contains(This) = False             neg : pos    =      1.6 : 1.0
            contains(an) = False             neg : pos    =      1.6 : 1.0
             contains(I) = False             pos : neg    =      1.4 : 1.0
             contains(I) = True              neg : pos    =      1.4 : 1.0
            contains(He) = False             pos : neg    =      1.2 : 1.0
            contains(My) = False             pos : neg    =      1.2 : 1.0
          contains(What) = False             neg : pos    =      1.2 : 1.0
         contains(about) = False             neg : pos    =      1.2 : 1.0
            contains(am) = False             pos : neg    =      1.2 : 1.0
       contains(amazing) = False             neg : pos    =      1.2 : 1.0
       contains(awesome) = False             neg : pos    =      1.2 : 1.0

<br>

We can now classify arbitrary text using the NaiveBayesClassifier.classify(text) method.

In [58]:
print(cl.classify("Their burgers are amazing"))  # "pos"
print(cl.classify("I don't like their pizza."))  # "neg"

pos
neg


In [59]:
# worth to mention, in many cases you may want to have also a "neutral" class
print(cl.classify("I liked the restaurant, the food was good, however the sevice was not good"))

neg


Another way to classify strings of text is to use TextBlob objects. You can pass classifiers into the constructor of a TextBlob.

TextBlob -> Super Cool package by the way


In [60]:
import textblob
blob = textblob.TextBlob("The beer was amazing. "
                "But the hangover was horrible. My boss was not happy.", classifier=cl)
print (blob)

The beer was amazing. But the hangover was horrible. My boss was not happy.


You can then call the classify() method on the blob.



In [61]:
blob.classify()  # "neg"

'neg'

You can also take advantage of TextBlob’s sentence tokenization and classify each sentence indvidually.

In [62]:
for sentence in blob.sentences:
    print(sentence)
    print(sentence.classify())
# "pos", "neg", "neg"

# Evaluate the classifier's accuracy on the held-out test set (returns a value between 0 and 1)
cl.accuracy(test)  

The beer was amazing.
pos
But the hangover was horrible.
neg
My boss was not happy.
neg


0.8333333333333334

We can improve our classifier by adding more training and test data. Here we’ll add data from the movie review corpus which was downloaded with NLTK.



In [63]:
nltk.download('movie_reviews')

import random
from nltk.corpus import movie_reviews

reviews = [(list(movie_reviews.words(fileid)), category)
              for category in movie_reviews.categories()
              for fileid in movie_reviews.fileids(category)]
random.shuffle(reviews)

new_train, new_test = reviews[0:300], reviews[301:400]

print('Review:' )
print(' '.join(new_train[0][0]))
print("\nSentiment: "+ new_train[0][1])

[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/luis/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


Review:
plot outline - wendy ( samantha press ) , a jazz singer , loves mack ( hugo race ) a criminal and wanna be rock singer who ' s planning a bank heist . mack is also being tailed by a couple of cops , one an inexperienced rookie ( dominic sweeney ) , the other ( john flaus ) a worn out veteran who frequents wendy ' s jazz club . they ' re tailing mack , because he has an audiotape that may show evidence of governmental corruption . meanwhile wendy ' s sexually awakener , fifteen year old sister ( rebecca elmaloglou ) has moved in with her , and is secretly watching mack and wendy ' s late night love trysts ? much zaniness ensues the review : main problem first - about 2 % of rood rock star to actor conversions in filmdom ever really work . unfortunately , trying to cast hugo race as a violent , sexy criminal falls into the " what the hell where they thinking " category that takes up 98 % of the rest . but , hell , it ' s not as if he ' s the only problem in this well - intentione

Let’s see what one of these documents looks like.



In [64]:
print(new_train[0])

(['plot', 'outline', '-', 'wendy', '(', 'samantha', 'press', ')', ',', 'a', 'jazz', 'singer', ',', 'loves', 'mack', '(', 'hugo', 'race', ')', 'a', 'criminal', 'and', 'wanna', 'be', 'rock', 'singer', 'who', "'", 's', 'planning', 'a', 'bank', 'heist', '.', 'mack', 'is', 'also', 'being', 'tailed', 'by', 'a', 'couple', 'of', 'cops', ',', 'one', 'an', 'inexperienced', 'rookie', '(', 'dominic', 'sweeney', ')', ',', 'the', 'other', '(', 'john', 'flaus', ')', 'a', 'worn', 'out', 'veteran', 'who', 'frequents', 'wendy', "'", 's', 'jazz', 'club', '.', 'they', "'", 're', 'tailing', 'mack', ',', 'because', 'he', 'has', 'an', 'audiotape', 'that', 'may', 'show', 'evidence', 'of', 'governmental', 'corruption', '.', 'meanwhile', 'wendy', "'", 's', 'sexually', 'awakener', ',', 'fifteen', 'year', 'old', 'sister', '(', 'rebecca', 'elmaloglou', ')', 'has', 'moved', 'in', 'with', 'her', ',', 'and', 'is', 'secretly', 'watching', 'mack', 'and', 'wendy', "'", 's', 'late', 'night', 'love', 'trysts', '?', 'much'

We can now update our classifier with the new training data using the update(new_data) method, as well as test it using the larger test dataset.



In [65]:
# Incrementally update the classifier with 300 movie reviews from NLTK,
# adding to what it already learned from the initial training data (no retraining from scratch)
cl.update(new_train) # it takes a while
accuracy = cl.accuracy(new_test) 
print("Accuracy: {0}".format(accuracy))

Accuracy: 0.7676767676767676


Here’s the full, updated script:



In [66]:
import re

train = [
    ('I love this sandwich.', 'pos'),
    ('This is an amazing place!', 'pos'),
    ('I feel very good about these beers.', 'pos'),
    ('This is my best work.', 'pos'),
    ("What an awesome view", 'pos'),
    ('I do not like this restaurant', 'neg'),
    ('I am tired of this stuff.', 'neg'),
    ("I can't deal with this", 'neg'),
    ('He is my sworn enemy!', 'neg'),
    ('My boss is horrible.', 'neg')
]
test = [
    ('The beer was good.', 'pos'),
    ('I do not enjoy my job', 'neg'),
    ("I ain't feeling dandy today.", 'neg'),
    ("I feel amazing!", 'pos'),
    ('Gary is a friend of mine.', 'pos'),
    ("I can't believe I'm doing this.", 'neg')
]

processed_features = []
def clean_text(data):
    for sentence in range(0, len(data)):  
        # Remove all the special characters
        processed_feature = re.sub(r'\W', ' ', str(data[sentence][0]))

        # remove all single characters
        processed_feature= re.sub(r'\s+[a-zA-Z]\s+', ' ', processed_feature)

        # Substituting multiple spaces with single space
        processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

        # Converting to Lowercase
        processed_feature = processed_feature.lower()

        data[sentence] = (processed_feature,data[sentence][1])
    return data

In [67]:
train_clean = clean_text(train)
test_clean = clean_text(test)

cl = NaiveBayesClassifier(train_clean)
accuracy = cl.accuracy(test_clean)
print("Accuracy: {0}".format(accuracy))

Accuracy: 0.8333333333333334


<br>

## 3.5 All together: News clustering example

In [68]:
import pandas as pd

# corpus of 120k news headlines, here shortened to 10k
all_news = pd.read_csv('../_datasets/news.csv')
all_news.head()

,news
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent."
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history."
2,"French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international"
3,"As many as 15,000 New Zealanders will be forced to find an alternative form of pain relief after the worldwide recall of the drug Vioxx, which has been found to double the risk of heart attacks and strokes."
4,"The group led by al Qaeda ally Abu Musab al-Zarqawi said it beheaded two Iraqi soldiers in broad daylight in Mosul, a statement found on an Islamist Web site on Friday said."


In [69]:
all_news.shape

(10000, 1)

In [70]:
all_news.iloc[2]

news    French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international 
Name: 2, dtype: object

In [71]:
all_news.iloc[2]["news"]

'French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international '

In [72]:
# same process as before, but for all lines
#tokenize, lowercase, remove punctuation
from nltk.tokenize import word_tokenize

def tokenizer_and_remove_punctuation(row):
  tokens = word_tokenize(row['news'])
  return [word.lower() for word in tokens if word.isalpha()]

all_news['tokenized'] = all_news.apply(tokenizer_and_remove_punctuation,axis=1)
all_news.head()

,news,tokenized
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.","[san, francisco, dell, said, thursday, its, profit, rose, percent, from, a, year, earlier, as, the, no, maker, boosted, sales, of, its, pcs, laptops, and, other, gear, by, percent]"
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history.","[american, phil, mickelson, registered, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, masters, champion, had, an, eagle, putt, on, the, for, a, record, but, missed, and, tapped, in, for, a, birdie, and, a, equalling, the, lowest, score, in, history]"
2,"French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintained, thursday, that, relations, between, their, countries, were, not, strained, by, their, disagreements, over, the, iraq, war, as, evidenced, by, their, cooperation, on, a, number, of, international]"
3,"As many as 15,000 New Zealanders will be forced to find an alternative form of pain relief after the worldwide recall of the drug Vioxx, which has been found to double the risk of heart attacks and strokes.","[as, many, as, new, zealanders, will, be, forced, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, has, been, found, to, double, the, risk, of, heart, attacks, and, strokes]"
4,"The group led by al Qaeda ally Abu Musab al-Zarqawi said it beheaded two Iraqi soldiers in broad daylight in Mosul, a statement found on an Islamist Web site on Friday said.","[the, group, led, by, al, qaeda, ally, abu, musab, said, it, beheaded, two, iraqi, soldiers, in, broad, daylight, in, mosul, a, statement, found, on, an, islamist, web, site, on, friday, said]"


In [73]:
# lemmatize with part of speech helpers
nltk.download('wordnet') # wordnet is the most well known lemmatizer for english
nltk.download('omw-1.4') # Open Multilingual Wordnet: extends wordnet with translations and synsets for many languages; required by some NLTK versions to avoid LookupError when lemmatizing
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# unfortunately pos_tag and lemmatize use different codes for parts of speech
def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper() # gets first letter of POS categorization
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN) # get returns second argument if first key does not exist

lemmatizer = WordNetLemmatizer()

def lemmatizer_with_pos(row):
  return [lemmatizer.lemmatize(word,get_wordnet_pos(word)) for word in row['tokenized']]

all_news['lemmatized'] = all_news.apply(lemmatizer_with_pos,axis=1)
all_news.head()


[nltk_data] Downloading package wordnet to /Users/luis/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/luis/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,news,tokenized,lemmatized
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.","[san, francisco, dell, said, thursday, its, profit, rose, percent, from, a, year, earlier, as, the, no, maker, boosted, sales, of, its, pcs, laptops, and, other, gear, by, percent]","[san, francisco, dell, say, thursday, it, profit, rise, percent, from, a, year, earlier, a, the, no, maker, boost, sale, of, it, pc, laptop, and, other, gear, by, percent]"
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history.","[american, phil, mickelson, registered, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, masters, champion, had, an, eagle, putt, on, the, for, a, record, but, missed, and, tapped, in, for, a, birdie, and, a, equalling, the, lowest, score, in, history]","[american, phil, mickelson, register, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, master, champion, have, an, eagle, putt, on, the, for, a, record, but, miss, and, tapped, in, for, a, birdie, and, a, equal, the, low, score, in, history]"
2,"French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintained, thursday, that, relations, between, their, countries, were, not, strained, by, their, disagreements, over, the, iraq, war, as, evidenced, by, their, cooperation, on, a, number, of, international]","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintain, thursday, that, relation, between, their, country, be, not, strain, by, their, disagreement, over, the, iraq, war, a, evidence, by, their, cooperation, on, a, number, of, international]"
3,"As many as 15,000 New Zealanders will be forced to find an alternative form of pain relief after the worldwide recall of the drug Vioxx, which has been found to double the risk of heart attacks and strokes.","[as, many, as, new, zealanders, will, be, forced, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, has, been, found, to, double, the, risk, of, heart, attacks, and, strokes]","[a, many, a, new, zealander, will, be, force, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, have, be, found, to, double, the, risk, of, heart, attack, and, stroke]"
4,"The group led by al Qaeda ally Abu Musab al-Zarqawi said it beheaded two Iraqi soldiers in broad daylight in Mosul, a statement found on an Islamist Web site on Friday said.","[the, group, led, by, al, qaeda, ally, abu, musab, said, it, beheaded, two, iraqi, soldiers, in, broad, daylight, in, mosul, a, statement, found, on, an, islamist, web, site, on, friday, said]","[the, group, lead, by, al, qaeda, ally, abu, musab, say, it, behead, two, iraqi, soldier, in, broad, daylight, in, mosul, a, statement, found, on, an, islamist, web, site, on, friday, say]"


In [74]:
# remove stopwords

def remove_sw(row):
  return list(set(row['lemmatized']).difference(stopwords.words()))

all_news['no_stopwords'] = all_news.apply(remove_sw,axis=1)
all_news.head()

,news,tokenized,lemmatized,no_stopwords
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.","[san, francisco, dell, said, thursday, its, profit, rose, percent, from, a, year, earlier, as, the, no, maker, boosted, sales, of, its, pcs, laptops, and, other, gear, by, percent]","[san, francisco, dell, say, thursday, it, profit, rise, percent, from, a, year, earlier, a, the, no, maker, boost, sale, of, it, pc, laptop, and, other, gear, by, percent]","[laptop, earlier, maker, boost, year, francisco, thursday, profit, percent, san, gear, rise, pc]"
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history.","[american, phil, mickelson, registered, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, masters, champion, had, an, eagle, putt, on, the, for, a, record, but, missed, and, tapped, in, for, a, birdie, and, a, equalling, the, lowest, score, in, history]","[american, phil, mickelson, register, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, master, champion, have, an, eagle, putt, on, the, for, a, record, but, miss, and, tapped, in, for, a, birdie, and, a, equal, the, low, score, in, history]","[golf, hawaii, history, grand, miss, win, master, tapped, kauai, phil, birdie, champion, record, slam, low, register, mickelson, putt, eagle, american, equal, score]"
2,"French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintained, thursday, that, relations, between, their, countries, were, not, strained, by, their, disagreements, over, the, iraq, war, as, evidenced, by, their, cooperation, on, a, number, of, international]","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintain, thursday, that, relation, between, their, country, be, not, strain, by, their, disagreement, over, the, iraq, war, a, evidence, by, their, cooperation, on, a, number, of, international]","[disagreement, country, international, president, cooperation, british, thursday, blair, jacques, maintain, number, minister, chirac, strain, prime, tony, french, iraq, evidence, relation]"
3,"As many as 15,000 New Zealanders will be forced to find an alternative form of pain relief after the worldwide recall of the drug Vioxx, which has been found to double the risk of heart attacks and strokes.","[as, many, as, new, zealanders, will, be, forced, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, has, been, found, to, double, the, risk, of, heart, attacks, and, strokes]","[a, many, a, new, zealander, will, be, force, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, have, be, found, to, double, the, risk, of, heart, attack, and, stroke]","[pain, vioxx, alternative, stroke, recall, double, relief, form, found, find, zealander, risk, attack, force, heart, worldwide]"
4,"The group led by al Qaeda ally Abu Musab al-Zarqawi said it beheaded two Iraqi soldiers in broad daylight in Mosul, a statement found on an Islamist Web site on Friday said.","[the, group, led, by, al, qaeda, ally, abu, musab, said, it, beheaded, two, iraqi, soldiers, in, broad, daylight, in, mosul, a, statement, found, on, an, islamist, web, site, on, friday, said]","[the, group, lead, by, al, qaeda, ally, abu, musab, say, it, behead, two, iraqi, soldier, in, broad, daylight, in, mosul, a, stateme

In [75]:
all_news.head(1)

,news,tokenized,lemmatized,no_stopwords
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.","[san, francisco, dell, said, thursday, its, profit, rose, percent, from, a, year, earlier, as, the, no, maker, boosted, sales, of, its, pcs, laptops, and, other, gear, by, percent]","[san, francisco, dell, say, thursday, it, profit, rise, percent, from, a, year, earlier, a, the, no, maker, boost, sale, of, it, pc, laptop, and, other, gear, by, percent]","[laptop, earlier, maker, boost, year, francisco, thursday, profit, percent, san, gear, rise, pc]"


In [76]:
# put all this cleaning together

def re_blob(row):
  return " ".join(row['no_stopwords'])

all_news['clean_blob'] = all_news.apply(re_blob,axis=1)
all_news.head()

,news,tokenized,lemmatized,no_stopwords,clean_blob
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.","[san, francisco, dell, said, thursday, its, profit, rose, percent, from, a, year, earlier, as, the, no, maker, boosted, sales, of, its, pcs, laptops, and, other, gear, by, percent]","[san, francisco, dell, say, thursday, it, profit, rise, percent, from, a, year, earlier, a, the, no, maker, boost, sale, of, it, pc, laptop, and, other, gear, by, percent]","[laptop, earlier, maker, boost, year, francisco, thursday, profit, percent, san, gear, rise, pc]",laptop earlier maker boost year francisco thursday profit percent san gear rise pc
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history.","[american, phil, mickelson, registered, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, masters, champion, had, an, eagle, putt, on, the, for, a, record, but, missed, and, tapped, in, for, a, birdie, and, a, equalling, the, lowest, score, in, history]","[american, phil, mickelson, register, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, master, champion, have, an, eagle, putt, on, the, for, a, record, but, miss, and, tapped, in, for, a, birdie, and, a, equal, the, low, score, in, history]","[golf, hawaii, history, grand, miss, win, master, tapped, kauai, phil, birdie, champion, record, slam, low, register, mickelson, putt, eagle, american, equal, score]",golf hawaii history grand miss win master tapped kauai phil birdie champion record slam low register mickelson putt eagle american equal score
2,"French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintained, thursday, that, relations, between, their, countries, were, not, strained, by, their, disagreements, over, the, iraq, war, as, evidenced, by, their, cooperation, on, a, number, of, international]","[french, president, jacques, chirac, and, british, prime, minister, tony, blair, maintain, thursday, that, relation, between, their, country, be, not, strain, by, their, disagreement, over, the, iraq, war, a, evidence, by, their, cooperation, on, a, number, of, international]","[disagreement, country, international, president, cooperation, british, thursday, blair, jacques, maintain, number, minister, chirac, strain, prime, tony, french, iraq, evidence, relation]",disagreement country international president cooperation british thursday blair jacques maintain number minister chirac strain prime tony french iraq evidence relation
3,"As many as 15,000 New Zealanders will be forced to find an alternative form of pain relief after the worldwide recall of the drug Vioxx, which has been found to double the risk of heart attacks and strokes.","[as, many, as, new, zealanders, will, be, forced, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, has, been, found, to, double, the, risk, of, heart, attacks, and, strokes]","[a, many, a, new, zealander, will, be, force, to, find, an, alternative, form, of, pain, relief, after, the, worldwide, recall, of, the, drug, vioxx, which, have, be, found, to, double, the, risk, of, heart, attack, and, stroke]","[pain, vioxx, alternative, stroke, recall, double, relief, form, found, find, zealander, risk, attack, force, heart, worldwide]",pain vioxx alternative stroke recall double relief form found find zealander risk attack force heart 

In [77]:
all_news.head(2)

,news,tokenized,lemmatized,no_stopwords,clean_blob
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.","[san, francisco, dell, said, thursday, its, profit, rose, percent, from, a, year, earlier, as, the, no, maker, boosted, sales, of, its, pcs, laptops, and, other, gear, by, percent]","[san, francisco, dell, say, thursday, it, profit, rise, percent, from, a, year, earlier, a, the, no, maker, boost, sale, of, it, pc, laptop, and, other, gear, by, percent]","[laptop, earlier, maker, boost, year, francisco, thursday, profit, percent, san, gear, rise, pc]",laptop earlier maker boost year francisco thursday profit percent san gear rise pc
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history.","[american, phil, mickelson, registered, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, masters, champion, had, an, eagle, putt, on, the, for, a, record, but, missed, and, tapped, in, for, a, birdie, and, a, equalling, the, lowest, score, in, history]","[american, phil, mickelson, register, a, to, win, the, grand, slam, of, golf, in, kauai, hawaii, the, master, champion, have, an, eagle, putt, on, the, for, a, record, but, miss, and, tapped, in, for, a, birdie, and, a, equal, the, low, score, in, history]","[golf, hawaii, history, grand, miss, win, master, tapped, kauai, phil, birdie, champion, record, slam, low, register, mickelson, putt, eagle, american, equal, score]",golf hawaii history grand miss win master tapped kauai phil birdie champion record slam low register mickelson putt eagle american equal score


In [78]:
#let's take only the most common 1000 words
from sklearn.feature_extraction.text import CountVectorizer
bow_vect = CountVectorizer(max_features=1000)

# fit creates one entry for each different word seen
X = bow_vect.fit_transform(all_news['clean_blob']).toarray()

In [79]:
as_df = pd.DataFrame(X,columns=bow_vect.get_feature_names_out())
as_df.head()

,abu,abuse,access,accord,account,accounting,accuse,acquire,acquisition,act,...,worth,wound,yahoo,yankee,yard,yasser,year,yesterday,york,young
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [80]:
all_news.iloc[2]["news"]

'French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international '

In [81]:
as_df.iloc[2][["abu", "abuse", "access", "minister", "president", "basketball"]]

abu           0
abuse         0
access        0
minister      1
president     1
basketball    0
Name: 2, dtype: int64

<br>

Now that each headline is represented as a 1000-dimensional word-count vector, we can cluster them using **KMeans**.

The idea is straightforward: headlines that use similar vocabulary should land in the same cluster, and those clusters should naturally correspond to news topics (e.g. sports, politics, business) — without us ever providing topic labels.

Here's what the next steps do:

1. **Fit KMeans with 6 clusters** — the algorithm assigns each headline to one of 6 groups based on the similarity of its word-count vector.
2. **Predict cluster labels** — each headline gets a cluster number (0–5).
3. **Inspect the clusters** — browse the headlines in each cluster and see whether a coherent topic emerges.

> **Note:** The number of clusters (`n_clusters=6`) is a hyperparameter we choose upfront. In practice you'd use methods like the *elbow method* or *silhouette score* to find a good value.


In [82]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=6,random_state=100)
kmeans.fit(X)
pred = kmeans.predict(X)

In [83]:
pred

array([5, 4, 3, ..., 4, 4, 0], dtype=int32)

In [84]:
predict_df = pd.concat([all_news['news'],pd.DataFrame(pred,columns=['class'])],axis=1)
predict_df.head()

,news,class
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.",5
1,"American Phil Mickelson registered a 59 to win the Grand Slam of Golf in Kauai, Hawaii. The Masters champion had an eagle putt on the 18th for a record 58 but missed and tapped in for a birdie and a 59, equalling the lowest score in stroke-play history.",4
2,"French President Jacques Chirac and British Prime Minister Tony Blair maintained Thursday that relations between their countries were not strained by their disagreements over the Iraq war, as evidenced by their cooperation on a number of international",3
3,"As many as 15,000 New Zealanders will be forced to find an alternative form of pain relief after the worldwide recall of the drug Vioxx, which has been found to double the risk of heart attacks and strokes.",4
4,"The group led by al Qaeda ally Abu Musab al-Zarqawi said it beheaded two Iraqi soldiers in broad daylight in Mosul, a statement found on an Islamist Web site on Friday said.",4


In [85]:
pd.set_option('display.max_colwidth', None)

<br>

Let's check all the news assigned to each cluster...

In [86]:
# News assigned to cluster 0
# ... many of them seem to be Sports News
predict_df[predict_df['class']==0]

,news,class
14,Everton striker Wayne Rooney says he is quot;disappointed quot; with the way the club have handled his transfer request. Newcastle and Manchester United want to sign the 18-year-old who requested a transfer on Friday,0
41,"Barnstable had the upperhand once again. The Red Raiders are one win away from defending their MIAA Division 1 state volleyball championship after blanking Chelmsford, 3-0, in the semifinals last night in Mansfield. It was the second time this fall Chelmsford had been frustrated by the powerful Barnstable lineup.",0
69,European governments on Monday spoke out against a French proposal that the European Union restrict development aid to poorer member states that seek to lure foreign investment with low corporate tax rates.,0
87,PHILADELPHIA -- The burst was back. Displaying some of the giddy-up he showed against Ball State -- before suffering a left knee injury on his 21st carry of a 129-yard performance -- L.V. Whitworth returned to his season-opening form yesterday. The redshirt freshman tailback rushed 17 times for a career-high 151 yards and touchdowns of 29 and 39 yards to ...,0
95,The United Nations has warned its staff\in Thailand to be careful following threats by a separatist\group to stage attacks in Bangkok in revenge for the deaths of\85 Muslim protesters in the south last week.,0
...,...,...
9898,"New York -- On a rainy Wednesday at the US Open, Serena Williams got a phone call from Arlen Kantarian, chief executive of the United States Tennis Association.",0
9920,"The United States would deal with\Iran as part of a group of Iraq's neighbors that will meet in\Egypt this month, although Washington has no diplomatic\relations with Tehran, Secretary of State Colin Powell said on\Tuesday.",0
9984,"McDonald's Corp. &lt;A HREF=""http://www.investor.reuters.com/FullQuote.aspx?ticker=MCD.N target=/stocks/quickinfo/fullquote""&gt;MCD.N&lt;/A&gt; said Monday that sales at its namesake hamburger restaurants open at least 13 months rose 6.1 percent in October, helped by a popular Monopoly game promotion in the United States.",0
9995,A business columnist at the Seattle Times in the United States has resigned after admitting he copied the work of other journalists.,0


In [87]:
# News assigned to cluster 1
# ... seem to be World News
predict_df[predict_df['class']==1]

,news,class
23,"Indonesian police on Saturday released security camera images of a truck bombing outside the Australian Embassy, and investigators found traces of explosives in a room rented by two Malaysian militants wanted in the blast. Also Saturday, around 1,000 members of a hardline Muslim group rallied in downtown Jakarta against Thursday's attack, which killed nine people, two of them suspected suicide bombers...",1
31,"JERUSALEM: Syria is directly involved in terrorism and will not be granted immunity by Israel, the deputy defence minister said on Monday, but stopped short of claiming formal responsibility for the killing of a Hamas leader in Damascus.",1
53,"Turkish television stations broadcast a video Friday that claimed that Habib Akdas, suspected leader of the Turkish al-Qaida cell blamed for November suicide bombings in Istanbul, was killed this week in a US bombing raid in Iraq.",1
115,"India News: Srinagar, Dec 5 : Twelve persons, including ten army troopers, were killed in a landmine explosion around midnight Saturday night in south Kashmir #39;s Pulwama district.",1
116,"US-LED forces attacked two Iraqi rebel strongholds yesterday, killing nearly two dozen insurgents in a town near the Syrian border and bombing targets in Fallujah for a third day.",1
...,...,...
9879,Hezbollah sent a reconnaissance drone into Israeli territory over northern Jewish settlements Sunday in the first hostile aerial incursion from Lebanon since a hang glider attack 17 years ago killed six soldiers.,1
9931,Iraqi environment minister Mishkat\Moumin said she survived a suicide car bomb attack in Baghdad\on Tuesday that killed four of her bodyguards.,1
9946,": Suspected Muslim insurgents attacked an army unit protecting Buddhist monks at a monastery early Friday, killing one of the soldiers as sectarian violence continued in southern Thailand, police said.",1
9949,Several workers are believed to have been killed and others injured after a contruction site collapsed at Dubai airport. The workers were trapped under rubble at the site of a \$4.,1


In [ ]:
# News assigned to cluster 4
# ... seem to be Sports / Racing 
predict_df[predict_df['class']==2]

,news,class
44,The race to save the British Grand Prix descended into acrimony with Formula 1 supremo Bernie Ecclestone issuing a libel writ against Jackie Stewart.,2
310,"South Africa #39;s Retief Goosen fired a seven-under-par 65 to take a one-shot lead over the season #39;s other three major winners after the first round of the 36-hole PGA Grand Slam of Golf in Kauai, Hawaii on Tuesday.",2
767,"Sete Gibernau has led a Telefonica Honda one-two in a dramatic first ever Qatar Grand Prix, which also saw Ruben Xaus take his first ever MotoGP podium and world championship leader Valentino Rossi crash out after a determined charge from the back of the",2
793,Rubens Barrichello leads home Michael Schumacher to seal a Ferrari one-two at the Italian Grand Prix.,2
1052,"New world number one Amelie Mauresmo struggled to a 7-5, 6-4 victory over Patty Schnyder in the second round of the Filderstadt Grand Prix on Thursday.",2
1522,World champion Michael Schumacher will lose ten grid positions on the Brazilian grand prix grid after crashing heavily in a shortened Saturday practice.,2
1881,The BRDC are looking to finalise a deal with Formula One Management (FOM) that will ensure that the British Grand Prix takes place next year on July 3rd and in 2005.,2
1887,Michael Schumacher won his 13th race of the season and first since August when he returned to his dominating form and captured the Japanese Grand Prix on Sunday.,2
1970,A former Irish priest famous for stunts that disrupted the marathon at the Athens Olympics and Britain's Grand Prix denied charges on Wednesday of indecency with a schoolgirl.,2
2116,Michael Schumacher wins a record 13th race in a season at the Japanese Grand Prix.,2


In [89]:
# News assigned to cluster 
# ... seem to be Business / Technology
predict_df[predict_df['class']==5]

,news,class
0,"SAN FRANCISCO (CBS.MW) -- Dell Inc. said Thursday its third-quarter profit rose 25 percent from a year earlier as the No. 1 personal-computer maker boosted sales of its PCs, laptops and other gear by 18 percent.",5
12,"Ballistic missile defence, a mere political hiccup south of the border even in an American presidential year, is showing surprising strength as a topic of public debate in Canada.",5
50,"Investment by businesses in foreign markets fell 18 percent in 2003 to \$560 billion as the global economy continued to struggle, but should improve this year as growth speeds up, the United Nations said Wednesday.",5
57,"Arsenal #39;s Thierry Henry, AC Milan #39;s Andriy Shevchenko, and Barcelona #39;s Ronaldinho are the three finalists for FIFA #39;s 2004 World Player of the Year award.",5
63,"Avon Products Inc., the world #39;s largest direct seller of cosmetics, reported its first US sales decline in five years, sending the company #39;s shares down the most since 2000.",5
...,...,...
9975,"Worldwide sales of computer chips are expected to set a record this year, according to the San Jose, Calif.-based trade group Semiconductor Industry Association.",5
9983,"TiVo Inc., maker of digital\television video recorders, will next year add ways for viewers\to see advertising and corporate logos even as they try to skip\commercials, the company said on Wednesday.",5
9987,"Semiconductor Manufacturing International Corp., a leading Chinese computer chip maker, plans to begin supplying chips using more advanced technology to major clients such as Texas Instruments Inc. beginning next year, the companies said Friday.",5
9988,The militant group is poised to sponsor candidates in legislative and municipal elections expected later next year.,5
